# Embedding Visualization — wav2vec2 phonological feature study

Exploratory look at the extracted hidden-state embeddings to build intuition **before** formal probing.

For every utterance we mean-pool each transformer layer over time → one vector per (utterance, layer). We then ask three questions:

1. **Do languages separate?** — 2D PCA / t-SNE projections, colored by language.
2. **Which layer carries the most language structure?** — silhouette score per layer.
3. **Do languages converge or diverge with depth?** — cosine similarity between language centroids across layers.

> ⚠️ Mean-pooling collapses a whole utterance into one point, so these plots reveal **utterance-level structure (language / speaker / channel)**, not phoneme-level phonological detail. They are for orientation; the phonological probing itself needs segment-level features (see notes at the end).

In [ ]:
import os, sys

# Run as if from the project root, regardless of where this notebook lives
if os.path.basename(os.getcwd()) == "notebook":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import gc
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from numpy.linalg import norm
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", context="notebook")

## Configuration

Point `EMBEDDINGS_DIR` at the folder your extraction jobs wrote to. The jobs save one pickle per language under a per-model subfolder: `embeddings/<model_tag>/<lang>_features.pkl`.

In [ ]:
# Root folder produced by the extraction jobs.
EMBEDDINGS_DIR = "embeddings"

MODELS = {
    "base":     os.path.join(EMBEDDINGS_DIR, "base"),
    "xlsr53":   os.path.join(EMBEDDINGS_DIR, "xlsr53"),
    "xlsr300m": os.path.join(EMBEDDINGS_DIR, "xlsr300m"),
}

LANGUAGES = ["en_us", "de_de", "es_419"]
LANG_COLORS = {"en_us": "#1f77b4", "de_de": "#ff7f0e", "es_419": "#2ca02c"}

# Keep only models whose folder actually exists on disk.
MODELS = {k: v for k, v in MODELS.items() if os.path.isdir(v)}
assert MODELS, f"No model folders found under '{EMBEDDINGS_DIR}'. Check EMBEDDINGS_DIR."
print("Models found:", list(MODELS))

## Load &amp; pool

Each pickle stores *every* layer at full time resolution, so the files are large. We therefore read each file **once**, mean-pool every layer over time, keep only the small pooled vectors, and discard the raw hidden states. The resulting `POOLED` cache feeds all plots below.

In [ ]:
def load_features(path):
    with open(path, "rb") as f:
        return pickle.load(f)


def pool_all_layers(model_dir):
    """Load each language file ONCE and mean-pool every hidden-state layer over
    time, so each utterance becomes one vector per layer.

    Returns (pooled, n_layers) where pooled[L] = {"X": (N, D), "lang": [...], "id": [...]}.
    Raw hidden states are released right after pooling to keep memory low.
    """
    pooled, n_layers = None, None
    for lang in LANGUAGES:
        path = os.path.join(model_dir, f"{lang}_features.pkl")
        if not os.path.exists(path):
            print(f"  (skipping missing file: {path})")
            continue
        data = load_features(path)
        for s in data:
            hs = s["hidden_states"]                      # list of (1, T, D) arrays
            if pooled is None:
                n_layers = len(hs)
                pooled = {L: {"X": [], "lang": [], "id": []} for L in range(n_layers)}
            for L in range(n_layers):
                pooled[L]["X"].append(hs[L].mean(axis=1).squeeze())   # (D,)
                pooled[L]["lang"].append(lang)
                pooled[L]["id"].append(s.get("id"))
        del data
        gc.collect()
    for L in pooled:
        pooled[L]["X"] = np.vstack(pooled[L]["X"])
    return pooled, n_layers


# Build the pooled cache once; every plot below reuses it (each pickle read only once).
POOLED, NLAYERS = {}, {}
for tag, mdir in MODELS.items():
    print(f"Loading {tag} ...")
    POOLED[tag], NLAYERS[tag] = pool_all_layers(mdir)
    X0 = POOLED[tag][0]["X"]
    print(f"  {tag}: {X0.shape[0]} utterances, dim={X0.shape[1]}, layers=0..{NLAYERS[tag]-1}")


def middle_layer(tag):
    return NLAYERS[tag] // 2

## 1. Do languages separate? (PCA &amp; t-SNE)

PCA shows global/linear structure; t-SNE shows local neighborhoods. Compare the three models side by side at a representative middle layer.

Read it carefully: coordinates are **not** comparable across models (different dims & training) — only the within-plot grouping matters. For t-SNE, cluster *positions and absolute distances are not meaningful*; only which points group together is.

In [ ]:
def scatter_2d(ax, XY, langs, title):
    langs = np.array(langs)
    for lg in LANGUAGES:
        m = langs == lg
        if m.any():
            ax.scatter(XY[m, 0], XY[m, 1], s=18, alpha=0.7,
                       label=lg, color=LANG_COLORS.get(lg))
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])


for method in ["PCA", "t-SNE"]:
    fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 4.5), squeeze=False)
    for ax, (tag, _) in zip(axes[0], MODELS.items()):
        L = middle_layer(tag)
        d = POOLED[tag][L]
        X = StandardScaler().fit_transform(d["X"])
        if method == "PCA":
            XY = PCA(n_components=2, random_state=0).fit_transform(X)
        else:
            perp = max(5, min(30, (len(X) - 1) // 3))
            XY = TSNE(n_components=2, init="pca", random_state=0,
                      perplexity=perp).fit_transform(X)
        scatter_2d(ax, XY, d["lang"], f"{tag}  (layer {L}/{NLAYERS[tag]-1})")
    axes[0][0].legend(loc="best", fontsize=8)
    fig.suptitle(f"{method}: mean-pooled utterance embeddings, colored by language", y=1.02)
    fig.tight_layout()
    plt.show()

## 2. Which layer separates languages best?

Silhouette score treats each language as a cluster: higher = languages are more separable at that layer. The curve tells you *where in the network* language identity is most strongly encoded — useful context for the layer-analysis probing experiment.

In [ ]:
rows = []
for tag in MODELS:
    for L in range(NLAYERS[tag]):
        d = POOLED[tag][L]
        X = StandardScaler().fit_transform(d["X"])
        y = np.array(d["lang"])
        if len(set(y)) > 1 and len(y) > len(set(y)):
            rows.append({"model": tag, "layer": L, "silhouette": silhouette_score(X, y)})
sep_df = pd.DataFrame(rows)

plt.figure(figsize=(8, 5))
for tag in MODELS:
    sub = sep_df[sep_df.model == tag]
    plt.plot(sub.layer, sub.silhouette, marker="o", label=tag)
plt.xlabel("Hidden-state layer")
plt.ylabel("Silhouette score by language")
plt.title("How strongly each layer separates the three languages")
plt.legend(); plt.tight_layout(); plt.show()

sep_df.pivot(index="layer", columns="model", values="silhouette").round(3)

## 3. Do languages converge or diverge across layers?

Mean cosine similarity between the three language centroids at each layer. Decreasing = the model pushes languages apart with depth; high/increasing = it represents them more similarly (more language-independent), which is what you would hope underlies good cross-lingual transfer.

The heatmap then zooms into one model/layer to see *which language pairs* are most alike.

In [ ]:
def cosine(a, b):
    return float(a @ b / (norm(a) * norm(b) + 1e-9))


def lang_centroids(d):
    y = np.array(d["lang"]); X = d["X"]
    return {lg: X[y == lg].mean(axis=0) for lg in LANGUAGES if (y == lg).any()}


rows = []
for tag in MODELS:
    for L in range(NLAYERS[tag]):
        cen = lang_centroids(POOLED[tag][L])
        names = list(cen)
        pairs = [cosine(cen[a], cen[b]) for i, a in enumerate(names) for b in names[i + 1:]]
        if pairs:
            rows.append({"model": tag, "layer": L, "mean_cos": float(np.mean(pairs))})
sim_df = pd.DataFrame(rows)

plt.figure(figsize=(8, 5))
for tag in MODELS:
    sub = sim_df[sim_df.model == tag]
    plt.plot(sub.layer, sub.mean_cos, marker="o", label=tag)
plt.xlabel("Hidden-state layer")
plt.ylabel("Mean inter-language cosine similarity")
plt.title("Do language centroids converge or diverge across layers?")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
HEATMAP_MODEL = next(iter(MODELS))      # change to any tag in MODELS
HEATMAP_LAYER = middle_layer(HEATMAP_MODEL)

cen = lang_centroids(POOLED[HEATMAP_MODEL][HEATMAP_LAYER])
names = list(cen)
M = np.array([[cosine(cen[a], cen[b]) for b in names] for a in names])

plt.figure(figsize=(5, 4))
sns.heatmap(M, xticklabels=names, yticklabels=names, annot=True, fmt=".2f",
            cmap="viridis", vmin=0, vmax=1)
plt.title(f"{HEATMAP_MODEL}: language-centroid cosine similarity "
          f"(layer {HEATMAP_LAYER}/{NLAYERS[HEATMAP_MODEL]-1})")
plt.tight_layout(); plt.show()

## Notes &amp; how this ties back to the project

- **What these views can / can't tell you.** They show how utterances group by *language* in each model and layer. They do **not** measure voicing / manner / place — those are phoneme-level and are washed out by mean-pooling.
- **Across models, compare patterns only** (separability curves, convergence trends), never raw coordinates — dims and training differ (base = 768-d / 13 layers; XLSR &amp; XLS-R = 1024-d / 25 layers).
- **t-SNE caveat:** with a few hundred points the layout shifts with perplexity/seed; treat it as qualitative.
- **Next step toward the real goal:** to visualize *phonological* structure, pool at the phoneme level (forced alignment → per-segment vectors) and color points by voicing / manner instead of language. The same three views then become genuinely phonological.

Optional extensions: color the 2D plots by speaker `gender` (FLEURS provides it) to check whether the structure is language or speaker; or plot per-layer PCA explained-variance to gauge intrinsic dimensionality.